In [ ]:
%matplotlib inline# ----- Section 0 -----


import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


#Opening the file from computer and changing titles to more suitable


df = pd.read_csv(
   "trips-abroad.txt",
   sep="\t",
   encoding="latin1"
)




df['Kuukausi'] = pd.to_datetime(df['Kuukausi'], format='%YM%m')
ts = (df.set_index('Kuukausi').asfreq("MS"))
print(ts.columns)
y = ts['Ulkomaanmatkat'].rename('Trips')


print(y.head())
print(y.index.min(), "→", y.index.max(), "| freq:", y.index.freqstr)


# Plotting the raw data


plt.figure()
plt.plot(y)
plt.title("Trips (Monthly)")
plt.xlabel("Date")
plt.ylabel("Trips (Thousands)")
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.show()


# ----- Section 1 -----
# Model identification


import numpy as np
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf




# --- ACF / PACF on the raw series ---
max_lags = 36 # 3 years of monthly lags


fig = plot_acf(y, lags=max_lags)
fig.suptitle("ACF — Trips (raw)")
plt.show()


fig = plot_pacf(y, lags=max_lags, method="ywm")
fig.suptitle("PACF — Trips (raw)")
plt.show()


# --- ACF / PACF on first-differenced series ---
y_diff1 = y.diff().dropna()


fig = plot_acf(y_diff1, lags=max_lags)
fig.suptitle("ACF — Trips (1st difference)")
plt.show()


fig = plot_pacf(y_diff1, lags=max_lags, method="ywm")
fig.suptitle("PACF — Trips (1st difference)")
plt.show()


# --- Seasonal difference at lag 12 for monthly seasonality ---
y_seasdiff = y.diff(12).dropna()


fig = plot_acf(y_seasdiff, lags=max_lags)
fig.suptitle("ACF — Trips (seasonal diff, lag=12)")
plt.show()


fig = plot_pacf(y_seasdiff, lags=max_lags, method="ywm")
fig.suptitle("PACF — Trips (seasonal diff, lag=12)")
plt.show()




# ----- Section 2 -----
# Model estimation


# Use log to stabilize variance
y_log = np.log(y)


# Time-based split (last 12 months as test)
h = 12
train = y_log.iloc[:-h]
test  = y_log.iloc[-h:]


print("Train:", train.index.min().date(), "→", train.index.max().date(), "| n=", len(train))
print("Test :", test.index.min().date(),  "→", test.index.max().date(),  "| n=", len(test))


plt.figure()
plt.plot(train, label="train (log)")
plt.plot(test, label="test (log)")
plt.title("Trips (log) — Train/Test Split")
ax = plt.gca()
ax.ticklabel_format(axis='y', style='plain')
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.legend()
plt.show()


import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox


model = sm.tsa.SARIMAX(
   train,
   order=(0, 1, 0),
   seasonal_order=(0, 1, 1, 12),
   enforce_stationarity=False,
   enforce_invertibility=False
)


res = model.fit(disp=False)
print(res.summary())




# ----- Section 3 -----
# Diagnostics






# Built-in diagnostic plots: standardized residuals, QQ, ACF of residuals, etc.
res.plot_diagnostics(figsize=(10, 12))
plt.show()


# Ljung-Box test for residual autocorrelation (want p-values not tiny)
lb = acorr_ljungbox(res.resid.dropna(), lags=[12, 24], return_df=True)
print(lb)




# ----- Section 4 -----
# Forecasting and evaluation




import sklearn as sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error


# Forecast h steps ahead
fc = res.get_forecast(steps=h)


fc_mean_log = fc.predicted_mean
fc_ci_log   = fc.conf_int()


# Back-transform to original scale
fc_mean = np.exp(fc_mean_log)
fc_lower = np.exp(fc_ci_log.iloc[:, 0])
fc_upper = np.exp(fc_ci_log.iloc[:, 1])


y_test_orig = np.exp(test)
y_train_orig = np.exp(train)


# Metrics on original scale
mae = mean_absolute_error(y_test_orig, fc_mean)
rmse = np.sqrt(mean_squared_error(y_test_orig, fc_mean))
mape = np.mean(np.abs((y_test_orig - fc_mean) / y_test_orig)) * 100


print(f"MAE : {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAPE: {mape:.2f}%")


plt.figure(figsize=(10, 4))
plt.plot(y_train_orig, label="Train")
plt.plot(y_test_orig, label="Test (actual)")
plt.plot(fc_mean, label="Forecast")
ax = plt.gca()
ax.ticklabel_format(axis='y', style='plain')
plt.fill_between(fc_mean.index, fc_lower, fc_upper, alpha=0.2, label="95% CI")
plt.title("SARIMA Forecast — Trips")
plt.legend()
plt.show()
